# 40. Does M0's design split into the groups of Lemma H.3?

Lemma H.3 (Appendix H.3) states that if the minutes split into **groups** that share no cell and lie at least the cooldown (240 minutes) apart, the inclusion constant falls from Γⁿ to Γ^(max_j n_j), where n_j is the number of alerts in group j. Here I check whether M0's development design forms a single group or several. This analysis is exploratory, and I run no model.

- Input: I use only `design_constant_v1/39_design.npz`, which notebook 39 saved (no prices; eligible minute stamps, months, deciles, alert counts per cell and M0 alert positions). Notebook 39's design has 1,414,298 eligible minutes, 800 alerts, 40 months, 262 cells with alerts and a 240-minute cooldown, and its M0 alerts match the cell counts and lie at least 240 minutes apart.
- Method: two eligible minutes a < b are linked by the cooldown if b − a < 240, and the minutes of one cell (month × decile) are linked through the cell's alert count; the groups of Lemma H.3 are the connected components of cells under these two relations. Only pairs of eligible minutes adjacent in time need checking, because a < b < c with c − a < 240 implies b − a < 240 and c − b < 240. I compute the components by two methods, scipy graph connected components and a pure-Python union-find (section 2, `1_groups.csv`). In section 3 I record, for each month boundary, the gap between the closest eligible minutes on either side (`2_month_boundaries.csv`).
- Reading, fixed in advance: a single group confirms the statement that M0's design is one group with n = 800; several groups mean that Lemma H.3 reduces the constant to Γ^(max_j n_j), which raises the trivial lower bound Γ* (α = 0.05) ≥ 1.795^(1/max_j n_j) accordingly. Either way, existing figures such as the upper bound Γ* ≤ 3.17 and the lower bound κ_D(1.5) ≥ 1.832 stay unchanged, and I report the same tables whatever the outcome.

I write all outputs to `design_groups_v1` and the summary to `40_SUMMARY.json`.

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: set DESIGN and OUT below to local paths.')

SPOT_BASE = Path('/content/drive/MyDrive/btc_lead_cost_results_77months_spot')
DESIGN = SPOT_BASE / 'design_constant_v1' / '39_design.npz'        # read only
OUT = SPOT_BASE / 'design_groups_v1'
LAMBDA_STAR = 1.795                       # notebook 35: worst-case p = 0.045 at this Lambda (lower end of the bracket)
EXPECT = dict(positions=1_414_298, alerts=800, months=40, cells_with_alerts=262, cooldown=240)   # notebook 39 (39_SUMMARY.json, Appendix H.7)
CHECK_EXPECT = True                       # set False only to test the notebook on a synthetic design
OUT.mkdir(parents=True, exist_ok=True)
print('design:', DESIGN, '| output:', OUT)

Mounted at /content/drive
design: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/design_constant_v1/39_design.npz | output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/design_groups_v1


In [2]:
# 1. Load the design and check that it is notebook 39's real design
z = np.load(DESIGN, allow_pickle=False)
t = z['t_min'].astype(np.int64)                 # eligible minutes (minute stamps), notebook-34 placement grid, zero-count cells dropped
mon = z['month_index'].astype(np.int64)         # 0 .. M-1
dec = z['decile'].astype(np.int64)              # 0 .. K-1
counts = z['counts'].astype(np.int64)           # alerts per (month, decile) cell
C = int(z['cooldown_min'])
m0 = z['m0_positions'].astype(np.int64)         # indices of M0's alerts among the eligible minutes
labels = z['month_labels']
M, K = counts.shape
if not np.all(np.diff(t) > 0):
    raise RuntimeError('Eligible minutes are not strictly increasing')
if not np.all(np.diff(mon) >= 0):
    raise RuntimeError('Months are not in time order')
cell = mon * K + dec
cells_used = np.unique(cell)
if np.any(counts.flat[cells_used] <= 0):
    raise RuntimeError('An eligible minute lies in a cell with no alert')
cnt_m0 = np.bincount(cell[m0], minlength=M * K).reshape(M, K)
if not np.array_equal(cnt_m0, counts):
    raise RuntimeError('M0 alerts do not match the cell counts')
if np.diff(t[np.sort(m0)]).min() < C:
    raise RuntimeError('Two M0 alerts are closer than the cooldown')
found = dict(positions=len(t), alerts=int(counts.sum()), months=M, cells_with_alerts=int((counts > 0).sum()), cooldown=C)
print('found   ', found)
print('expected', EXPECT)
if CHECK_EXPECT and found != EXPECT:
    raise RuntimeError('This is not the design of notebook 39')
print('OK: notebook-39 design; M0 alerts match the cell counts and keep the cooldown')

found    {'positions': 1414298, 'alerts': 800, 'months': 40, 'cells_with_alerts': 262, 'cooldown': 240}
expected {'positions': 1414298, 'alerts': 800, 'months': 40, 'cells_with_alerts': 262, 'cooldown': 240}
OK: notebook-39 design; M0 alerts match the cell counts and keep the cooldown


In [3]:
# 2. Groups of Lemma H.3: connected components of cells, linked by a shared cell or by eligible minutes less than C apart
t0 = time.time()
close = np.flatnonzero(np.diff(t) < C)                       # neighbouring eligible minutes that constrain each other
a, b = cell[close], cell[close + 1]
G = coo_matrix((np.ones(len(a)), (a, b)), shape=(M * K, M * K))
ncomp, lab = connected_components(G, directed=False)
lab_used = lab[cells_used]
_, grp = np.unique(lab_used, return_inverse=True)            # group index per used cell
n_groups = int(grp.max()) + 1

# second method: plain union-find over the same pairs
parent = list(range(M * K))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for u, v in zip(a.tolist(), b.tolist()):
    ru, rv = find(u), find(v)
    if ru != rv: parent[rv] = ru
roots = np.array([find(int(c)) for c in cells_used])
same = all(len(set(roots[grp == g])) == 1 for g in range(n_groups)) and len(set(roots)) == n_groups
if not same:
    raise RuntimeError('The two component computations disagree')

rows = []
for g in range(n_groups):
    cs = cells_used[grp == g]
    ms = np.unique(cs // K)
    sel = np.isin(cell, cs)
    rows.append(dict(group=g, cells=len(cs), months=len(ms), first_month=str(labels[ms.min()]), last_month=str(labels[ms.max()]),
                     eligible_minutes=int(sel.sum()), alerts=int(counts.flat[cs].sum())))
GR = pd.DataFrame(rows).sort_values('alerts', ascending=False).reset_index(drop=True)
GR.to_csv(OUT / '1_groups.csv', index=False)
print(f'{n_groups} group(s); both methods agree ({time.time() - t0:.1f}s)')
display(GR.head(20))

20 group(s); both methods agree (0.9s)


,group,cells,months,first_month,last_month,eligible_minutes,alerts
0,4,46,6,24285,24290,224945,120
1,1,36,5,24278,24282,173180,99
2,6,30,5,24292,24296,175327,99
3,8,20,3,24297,24299,97970,60
4,19,14,3,24314,24316,101631,60
5,13,21,3,24306,24308,116593,60
6,3,13,2,24283,24284,82631,40
7,9,14,2,24300,24301,74537,40
8,10,10,2,24302,24303,51218,40
9,14,10,2,24309,24310,65806,38


In [4]:
# 3. Month boundaries: smallest gap between an eligible minute of month m and one of month m + 1
brow = []
for m in range(M - 1):
    tm, tn = t[mon == m], t[mon == m + 1]
    gap = int(tn.min() - tm.max())
    brow.append(dict(boundary=f'{labels[m]}|{labels[m + 1]}', last_minute_gap=gap, linked_by_cooldown=gap < C))
BD = pd.DataFrame(brow)
BD.to_csv(OUT / '2_month_boundaries.csv', index=False)
print(f'month boundaries linked by the cooldown: {int(BD.linked_by_cooldown.sum())} of {len(BD)}')
display(BD[~BD.linked_by_cooldown]) if (~BD.linked_by_cooldown).any() else print('every month is linked to the next')

month boundaries linked by the cooldown: 23 of 39


,boundary,last_minute_gap,linked_by_cooldown
0,24277|24278,291,False
5,24282|24283,1450,False
7,24284|24285,421,False
13,24290|24291,1851,False
14,24291|24292,354,False
19,24296|24297,2194,False
22,24299|24300,1938,False
24,24301|24302,882,False
26,24303|24304,551,False
27,24304|24305,624,False


In [5]:
# 4. Summary and reading fixed in advance
nmax = int(GR.alerts.max())
summary = dict(design=str(DESIGN), found=found, groups=n_groups, alerts_largest_group=nmax,
               alerts_per_group=GR.alerts.tolist()[:20],
               month_boundaries_linked=int(BD.linked_by_cooldown.sum()), month_boundaries=len(BD),
               lemma_H3_exponent=nmax, trivial_lower_bound_gamma_star=LAMBDA_STAR ** (1 / nmax),
               reading=('single group: the design is one group with n = 800' if n_groups == 1 else
                        f'{n_groups} groups: Lemma H.3 gives Lambda = Gamma^{nmax}; the trivial bound becomes '
                        f'Gamma*_0.05 >= 1.795^(1/{nmax}) = {LAMBDA_STAR ** (1 / nmax):.5f}'))
(OUT / '40_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))

{
  "design": "/content/drive/MyDrive/btc_lead_cost_results_77months_spot/design_constant_v1/39_design.npz",
  "found": {
    "positions": 1414298,
    "alerts": 800,
    "months": 40,
    "cells_with_alerts": 262,
    "cooldown": 240
  },
  "groups": 20,
  "alerts_largest_group": 120,
  "alerts_per_group": [
    120,
    99,
    99,
    60,
    60,
    60,
    40,
    40,
    40,
    38,
    20,
    20,
    20,
    20,
    20,
    20,
    20,
    2,
    1,
    1
  ],
  "month_boundaries_linked": 23,
  "month_boundaries": 39,
  "lemma_H3_exponent": 120,
  "trivial_lower_bound_gamma_star": 1.0048869441996442,
  "reading": "20 groups: Lemma H.3 gives Lambda = Gamma^120; the trivial bound becomes Gamma*_0.05 >= 1.795^(1/120) = 1.00489"
}
